<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/04_multiphase_and_fields.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 4 of 7: Effective Diffusivity and Field Visualisation

*OpenImpala Tutorial Series — From first solve to HPC deployment*

---

OpenImpala provides two approaches to computing transport properties:

1. **Tortuosity solver** (`TortuosityHypre`) — Solves $\nabla \cdot (D\,\nabla\varphi) = 0$ with Dirichlet boundary conditions and computes $\tau$ from the resulting flux. This is what the high-level `oi.tortuosity()` function uses.
2. **Effective diffusivity solver** (`EffectiveDiffusivityHypre`) — Solves the periodic cell problem $\nabla \cdot (D\,\nabla\chi) = -\nabla \cdot (D\,\hat{e})$ to compute the effective diffusivity tensor via homogenisation.

Both give consistent results on binary structures, but the cell-problem approach generalises naturally to multi-phase composites with spatially varying $D(\mathbf{x})$.

This tutorial uses the `openimpala.core` API to run the effective diffusivity solver and visualise the solved field.

**What you will learn:**
1. Use the `openimpala.core` module directly (lower-level than `oi.tortuosity()`).
2. Solve the effective diffusivity cell problem on a binary structure.
3. Visualise the corrector field using AMReX plotfiles and `yt`.
4. Compare the result with the tortuosity approach.

**Prerequisites:** [Tutorial 1](01_hello_openimpala.ipynb) for the high-level API. [Tutorial 2](02_digital_twin.ipynb) introduced `openimpala.core` and `yt` visualisation — we build on that here.

In [ ]:
# Install OpenImpala, PoreSpy, yt (for AMReX plotfile visualisation), and Matplotlib.
!pip install -q openimpala porespy yt matplotlib

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import porespy as ps
import yt

import openimpala as oi
from openimpala import core as oic  # Low-level C++ API (pybind11 wrappers)

print(f"OpenImpala version {oi.__version__} loaded.")

## 1. Generating a Test Microstructure

We use PoreSpy to create a random porous structure with an anisotropic pore network. The structure is elongated in the Z-direction, so we expect lower tortuosity (easier transport) along Z than along X or Y.

In [ ]:
N = 64
np.random.seed(42)

micro = ps.generators.blobs(
    shape=[N, N, N],
    porosity=0.50,
    blobiness=1.5,
).astype(np.int32)

print(f"Microstructure: {micro.shape}, porosity = {micro.mean():.4f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, axis, label in zip(axes, [0, 1, 2], ["X", "Y", "Z"]):
    slc = np.take(micro, N // 2, axis=axis)
    ax.imshow(slc, cmap="gray", interpolation="nearest")
    ax.set_title(f"Mid-plane slice ({label})")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Solving with the Core API

The high-level `oi.tortuosity()` function wraps several steps (VoxelImage creation, percolation check, volume fraction, solver construction). Here we perform those steps explicitly using `openimpala.core`, which gives more control — for example, enabling plotfile output.

We solve the **effective diffusivity cell problem** in the Z-direction with `write_plotfile=True` so we can visualise the corrector field afterwards.

In [ ]:
out_dir = "./effdiff_plotfiles"
os.makedirs(out_dir, exist_ok=True)

with oi.Session():
    # 1. Convert NumPy array to VoxelImage
    img = oic.VoxelImage.from_numpy(
        np.ascontiguousarray(micro, dtype=np.int32), max_grid_size=32
    )

    # 2. Compute volume fraction for reference
    vf_calc = oic.VolumeFraction(img, 1, 0)
    vf = vf_calc.value_vf()
    print(f"Volume fraction (phase 1): {vf:.4f}")

    # 3. Solve the effective diffusivity cell problem (Z-direction)
    print("\nSolving cell problem (EffectiveDiffusivityHypre, Z-direction)...")
    eff_solver = oic.EffectiveDiffusivityHypre(
        img,
        phase_id=1,
        dir=oic.Direction.Z,
        solver_type=oic.EffDiffSolverType.FlexGMRES,
        results_path=out_dir,
        verbose=1,
        write_plotfile=True,
    )
    converged = eff_solver.solve()
    print(f"Converged: {converged} ({eff_solver.iterations} iterations, "
          f"residual = {eff_solver.residual_norm:.2e})")

    # 4. For comparison, also solve with TortuosityHypre
    print("\nSolving with TortuosityHypre for comparison...")
    tau_solver = oic.TortuosityHypre(
        img, vf, 1, oic.Direction.Z, oic.SolverType.FlexGMRES,
        out_dir, 0.0, 1.0, 1, False,
    )
    tau = tau_solver.value()
    print(f"Tortuosity (TortuosityHypre): {tau:.4f}")

## 3. Visualising the Corrector Field

The effective diffusivity solver writes the corrector field $\chi_z$ to an AMReX plotfile. We load it with [yt](https://yt-project.org/) and plot a slice. Regions where the field deviates strongly from zero indicate where the microstructure forces the diffusion flux to deviate from the imposed direction — this is the physical origin of tortuosity.

In [ ]:
# Find the generated plotfile directory
plotfile_dirs = [d for d in glob.glob(f"{out_dir}/*") if os.path.isdir(d)]
if plotfile_dirs:
    latest_plotfile = sorted(plotfile_dirs)[-1]
    print(f"Loading plotfile: {latest_plotfile}")

    yt.funcs.mylog.setLevel(40)  # Suppress verbose yt logging
    ds = yt.load(latest_plotfile)

    # List available fields to find the corrector field name
    field_list = [f for f in ds.field_list if f[0] == "boxlib"]
    print(f"Available fields: {field_list}")

    # Plot a slice through the Y-midplane
    field_name = field_list[0]  # Use the first available boxlib field
    slc = yt.SlicePlot(ds, normal="y", fields=field_name)
    slc.set_log(field_name, False)
    slc.set_cmap(field_name, "RdBu_r")
    slc.annotate_title(f"Corrector Field (Z-direction)")
    slc.show()
else:
    print("No plotfile found — check that write_plotfile=True was set.")

## 4. Extension: Multi-Phase Composites

The effective diffusivity solver supports spatially varying diffusion coefficients, which is essential for multi-phase materials such as battery electrodes with a carbon-binder domain (CBD). In that case, each phase is assigned its own diffusivity (e.g. $D_\text{pore} = 1.0$, $D_\text{CBD} = 0.1$, $D_\text{solid} = 0$).

Per-phase diffusivities are configured via AMReX's parameter database. When using the C++ executable, this is done through the input file:

```
tortuosity.active_phases = 1 2
tortuosity.phase_diffusivities = 1.0 0.1
```

For a worked example, see the test input file `tests/inputs/tMultiPhaseTransport.inputs` in the OpenImpala repository.

## Next Steps

This tutorial demonstrated the effective diffusivity cell-problem solver and how to visualise the internal field. The same `openimpala.core` API gives access to solver diagnostics (convergence, residual, iteration count) that the high-level facade summarises automatically.

**Continue the series:**
- [Tutorial 5: Surrogate Modelling](05_surrogate_modelling.ipynb) — Generate labelled datasets for machine learning.
- [Tutorial 6: Topology Optimisation](06_topology_optimisation.ipynb) — Use OpenImpala as a cost-function evaluator in an optimisation loop.
- [Tutorial 7: Scaling to HPC](07_hpc_scaling.ipynb) — Run on larger datasets with MPI.

---

## References & Further Reading

1. **OpenImpala:** S. Mayner, F. Ciucci, *OpenImpala: open-source computational framework for microstructural analysis of 3D tomography data*, SoftwareX (2024). [GitHub](https://github.com/BASE-Laboratory/OpenImpala)
2. **Homogenisation theory:** S. Torquato, *Random Heterogeneous Materials: Microstructure and Macroscopic Properties*, Springer (2002). Chapter 17 covers the cell problem for effective conductivity.
3. **Effective diffusivity in electrodes:** L. Holzer et al., *Microstructure–property relationships in a gas diffusion layer (GDL) for Polymer Electrolyte Fuel Cells, Part I: effect of compression and anisotropy of dry GDL*, Electrochimica Acta 227, 419–434 (2017). [doi:10.1016/j.electacta.2017.01.030](https://doi.org/10.1016/j.electacta.2017.01.030)
4. **Carbon-binder domain modelling:** F. L. E. Usseglio-Viretta et al., *Resolving the discrepancy in tortuosity factor estimation for Li-ion battery electrodes through micro-macro modeling and experiment*, J. Electrochem. Soc. 165(14), A3403 (2018). [doi:10.1149/2.0731814jes](https://doi.org/10.1149/2.0731814jes)
5. **yt for AMReX data:** M. J. Turk et al., *yt: A multi-code analysis toolkit for astrophysical simulation data*, Astrophys. J. Suppl. 192, 9 (2011). [doi:10.1088/0067-0049/192/1/9](https://doi.org/10.1088/0067-0049/192/1/9)